# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(file_path)

target_col = 'ctr'

# Define our full feature sets
numeric_features = [
    'impressions_90d', 'avg_position', 'content_age_days',
    'word_count', 'sessions_90d', 'pageviews_90d'
]

categorical_features = [
    'content_type', 'main_intent'
]

# 1. Handle missing values
# Fill in missing numeric values with 0 
for col in numeric_features:
    df[col] = df[col].fillna(0)

# Fill in missing categorical values with Unknown
for col in categorical_features:
    df[col] = df[col].fillna('Unknown')

# 2. Build the initial feature dataframe (dropping rows where target ctr is missing)
feature_df = df.dropna(subset=[target_col]).copy()

# 3. Categorical handling 
X = pd.get_dummies(feature_df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
y = feature_df[target_col]

print(f"Final Feature Vector Shape: {X.shape}")
display(X.head())

Final Feature Vector Shape: (30000, 12)


,impressions_90d,avg_position,content_age_days,word_count,sessions_90d,pageviews_90d,content_type_feedly article,content_type_keyword article,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional
0,3803,10.6,187,3221.0,17,22,False,True,False,False,False,True
1,15320,20.3,445,2481.0,9,10,False,True,False,True,False,False
2,12581,36.5,141,3515.0,11,14,False,True,False,True,False,False
3,11751,6.2,463,0.0,78,87,False,True,True,False,False,False
4,19140,44.0,263,2803.0,145,177,False,True,False,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

#### impressions 90d / sessions_90d / pageviews_90d 
numeric features, 90 days of rolling sum of visibility and traffic. Missing values filled with 0. Available before prediction.

#### avg_position 
numeric feature, average ranking position over 90 days. Missing values filled with 0. Available before prediction

#### content_age_days
numeric feature, age of the urls in days. Missing values filled with 0. Available before prediction.

#### word_count
numeric feature, length of the content. Missing values filled with 0. Available before prediction.

#### content_type / main_intent
categorical features, text descriptors of the page's purpose. Missing values filled with Unknown and one-hot coded in binary columns. Available before prediction.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Split the clean data 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the honest model
rf_honest = RandomForestRegressor(random_state=42, n_estimators=20, max_depth=5)
rf_honest.fit(X_train, y_train)
honest_rmse = np.sqrt(mean_squared_error(y_test, rf_honest.predict(X_test)))

# Leaked data (w/ clicks_90d)
X_leaked_train = X_train.copy()
X_leaked_test = X_test.copy()
X_leaked_train['leak_clicks_90d'] = feature_df.loc[X_train.index, 'clicks_90d']
X_leaked_test['leak_clicks_90d'] = feature_df.loc[X_test.index, 'clicks_90d']

# Train leaked model
rf_leaked = RandomForestRegressor(random_state=42, n_estimators=20, max_depth=5)
rf_leaked.fit(X_leaked_train, y_train)
leaked_rmse = np.sqrt(mean_squared_error(y_test, rf_leaked.predict(X_leaked_test)))

print(f"Honest model rmse: {honest_rmse:.4f}")
print(f"Leaked model rmse: {leaked_rmse:.4f}")
print("Result: 'clicks__90d' causes massive leakage and artificial accuracy.")

Honest model rmse: 3.0457
Leaked model rmse: 0.7259
Result: 'clicks__90d' causes massive leakage and artificial accuracy.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

#### clicks_90d, clicks_last30d, clicks_prev30d
excluded because our target is ctr (clicks/impressions). Including raw clicks leaks the mathematical answer directly to the model.

#### health_score, priority_score, action_type 
excluded because these are product decision flags and not observable signals

#### trend_pct, trend_direction
excluded from features because they are derived outcome buckets rather than pure observable signals, which risks creating circular logic where the model learns predefined thresholds rather than raw performance patterns.

## Self-check

Before you submit, confirm each line honestly:

- [/] Every section above is filled — markdown thinking AND the code that backs it
- [/] The notebook runs top to bottom with no errors (Runtime → Run all)
- [/] No client names, URLs, or private queries anywhere
- [/] My claims use careful words: observed, measured, directional, decision-support
- [/] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.